# 제곱 함수와 n 제곱 함수 만들기

In [1]:
def my_sq(x):
    return x ** 2

def my_exp(x, n):
    return x ** n

print(my_sq(4))
print(my_exp(2, 4))

16
16


# 시리즈와 apply 메서드

In [2]:
import pandas as pd

df = pd.DataFrame({'a': [10, 20, 30], 'b': [20, 30, 40]}) 
print(df)

print(df['a'] ** 2)

sq = df['a'].apply(my_sq) 
print(sq)

ex = df['a'].apply(my_exp, n=2) 
print(ex)

ex = df['a'].apply(my_exp, n=3) 
print(ex)

    a   b
0  10  20
1  20  30
2  30  40
0    100
1    400
2    900
Name: a, dtype: int64
0    100
1    400
2    900
Name: a, dtype: int64
0    100
1    400
2    900
Name: a, dtype: int64
0     1000
1     8000
2    27000
Name: a, dtype: int64


# 데이터 프레임과 apply 메서드

In [4]:
df = pd.DataFrame({'a': [10, 20, 30], 'b': [20, 30, 40]}) 
print(df)

def print_me(x): 
    print(x)

print(df.apply(print_me, axis=0)) #열 방향으로 함수 적용
print(df['a'])
print(df['b'])

def avg_3(x, y, z):
    return (x + y + z) / 3

#오류 print(df.apply(avg_3))

def avg_3_apply(col):
    x = col[0] 
    y = col[1] 
    z = col[2] 
    return (x + y + z) / 3

print(df.apply(avg_3_apply))

def avg_3_apply(col):
    sum = 0
    for item in col:
         sum += item
    return sum / df.shape[0]

def avg_2_apply(row):
    sum = 0
    for item in row:
        sum += item
    return sum / df.shape[1]   #행 방향으로 데이터 처리

print(df.apply(avg_2_apply, axis = 1))

    a   b
0  10  20
1  20  30
2  30  40
0    10
1    20
2    30
Name: a, dtype: int64
0    20
1    30
2    40
Name: b, dtype: int64
a    None
b    None
dtype: object
0    10
1    20
2    30
Name: a, dtype: int64
0    20
1    30
2    40
Name: b, dtype: int64
a    20.0
b    30.0
dtype: float64
0    15.0
1    25.0
2    35.0
dtype: float64


# 데이터프레임의 누락값을 처리한 다음 apply 메서드 사용하기 - 열 방향

In [8]:
import seaborn as sns

titanic = sns.load_dataset("titanic")
print(titanic.info())

import numpy as np

#누락값의 개수를 반환하는 함수
def count_missing(vec):
    null_vec = pd.isnull(vec)
    null_count = np.sum(null_vec)
    return null_count

cmis_col = titanic.apply(count_missing)
print(cmis_col)

#누락값의 비율을 계산하는 함수
def prop_missing(vec):
    num = count_missing(vec)
    dem = vec.size   #데이터프레임의 전체 데이터수
    return num / dem

pmis_col = titanic.apply(prop_missing)
print(pmis_col)

#누락값이 아닌 데이터의 비율
def prop_complete(vec):
    return 1 - prop_missing(vec)

pcom_col = titanic.apply(prop_complete)
print(pcom_col)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
None
survived         0
pclass           0
sex              0
age            17

# 데이터프레임의 누락값을 처리한 다음 apply 메서드 사용하기 - 행 방뱡

In [11]:
cmis_row = titanic.apply(count_missing, axis=1)
pmis_row = titanic.apply(prop_missing, axis=1)
pcom_row = titanic.apply(prop_complete, axis=1)

print(cmis_row.head())
print(pmis_row.head())
print(pcom_row.head())

titanic['num_missing'] = titanic.apply(count_missing, axis=1)

print(titanic.head())
print(titanic.loc[titanic.num_missing > 1, :].sample(10))  #누락값 2개 이상 데이터 추출

0    1
1    0
2    1
3    0
4    1
dtype: int64
0    0.0625
1    0.0000
2    0.0625
3    0.0000
4    0.0625
dtype: float64
0    0.9375
1    1.0000
2    0.9375
3    1.0000
4    0.9375
dtype: float64
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  num_missing  
0    man        True  NaN  Southampton    no  False            1  
1  woman       False    C    Cherbourg   yes  False            0  
2  woman       False  NaN  Southampton   yes   True            1  
3  woman       False    C  Southampton   yes  False            0  
4    man  